# Store Intelligence Report

**Daily store-level insights: 2–3 key metrics per domain (Sales, Labor, Inventory), risk/opportunity detection, and an AI-generated executive summary.**


In [ ]:
# ── Cell 1: Parameters ────────────────────────────────────────────────
store_id = "STR000001"           # Change to target store ID
lookback_days = 7                # Analysis window (days)

In [ ]:
# ── Imports & Configuration ────────────────────────────────────────────
import json
from datetime import datetime, timedelta

import pandas as pd
from pyspark.sql import functions as F

# Risk thresholds
REVENUE_CRITICAL_DROP = 15.0
REVENUE_WARNING_DROP  = 5.0

# Date boundaries
today = datetime.strptime('2026-01-17', '%Y-%m-%d').date()
period_end   = today
period_start = today - timedelta(days=lookback_days)
prior_end    = period_start - timedelta(days=1)
prior_start  = prior_end - timedelta(days=lookback_days - 1)

print(f"Analysis period: {period_start} → {period_end}")
print(f"Prior period:    {prior_start} → {prior_end}")

In [ ]:
# ── Cell 3: Store Validation ──────────────────────────────────────────
# Note: The Lakehouse catalog name contains a hyphen (5-in-5), so we must quote it with backticks.
# The fully qualified path is: `5-in-5`.lh_retaildata.retail.vw_sales_summary

store_info_df = spark.read.synapsesql("lh_retaildata.retail.vw_sales_summary") \
    .filter(f"str_id = '{store_id}'") \
    .select("str_id", "str_name", "str_type", "reg_id", "reg_name") \
    .distinct() \
    .limit(1)

if store_info_df.count() == 0:
    all_stores = spark.read.synapsesql("lh_retaildata.retail.vw_sales_summary") \
        .filter(f"str_id = '{store_id}'") \
        .distinct() \
        .sort(str_id)

    all_stores = all_stores.toPandas()
    store_list = "\n".join(f"  • {r['str_id']}: {r['str_name']}" for _, r in all_stores.iterrows())
    raise ValueError(
        f"Store ID {store_id} not found. Available stores:\n" + store_list
    )

store_row = store_info_df.first()
STORE_ID   = store_row["str_id"]
STORE_NAME = store_row["str_name"]
STORE_TYPE = store_row["str_type"]
REG_ID     = store_row["reg_id"]
REG_NAME   = store_row["reg_name"]

print(f"✓ Store resolved: {STORE_NAME} (ID: {STORE_ID})")
print(f"  Type: {STORE_TYPE} | Region: {REG_NAME} (ID: {REG_ID})")

In [ ]:
# ── Sales Metrics ──────────────────────────────────────────────────────

# Current & prior period sales
sales_current = spark.read.synapsesql("lh_retaildata.retail.vw_sales_summary") \
    .filter(f"str_id = '{STORE_ID}' AND order_date BETWEEN '{period_start}' AND '{period_end}'") \
    .toPandas()

sales_prior = spark.read.synapsesql("lh_retaildata.retail.vw_sales_summary") \
    .filter(f"str_id = '{STORE_ID}' AND order_date BETWEEN '{prior_start}' AND '{prior_end}'") \
    .toPandas()

# Peer benchmark
region_peers = spark.read.synapsesql("lh_retaildata.retail.vw_sales_summary") \
    .filter(f"reg_id = '{REG_ID}' AND str_type = '{STORE_TYPE}' AND order_date BETWEEN '{period_start}' AND '{period_end}'") \
    .groupBy("str_id") \
    .agg(F.sum("line_total").alias("revenue")) \
    .toPandas()

# KPI 1: Revenue + WoW change
revenue = round(sales_current["line_total"].sum(), 2) if not sales_current.empty else 0
revenue_prior = round(sales_prior["line_total"].sum(), 2) if not sales_prior.empty else 0
rev_wow = round((revenue - revenue_prior) / revenue_prior * 100, 1) if revenue_prior else None

# KPI 2: Orders
orders = int(sales_current["ord_id"].nunique()) if not sales_current.empty else 0

# KPI 3: Peer rank
peer_avg_revenue = round(region_peers["revenue"].mean(), 2) if not region_peers.empty else 0
peer_rank = int((region_peers["revenue"] >= revenue).sum()) if not region_peers.empty else 0
peer_count = len(region_peers)

sales_kpis = {"revenue": revenue, "revenue_wow": rev_wow, "orders": orders,
              "peer_rank": peer_rank, "peer_count": peer_count, "peer_avg_revenue": peer_avg_revenue}

print(f"Revenue: ${revenue:,.0f} (WoW: {rev_wow}%)")
print(f"Orders:  {orders}")
print(f"Peer rank: #{peer_rank} of {peer_count} ({STORE_TYPE} in {REG_NAME})")

In [ ]:
# ── Labor Metrics ──────────────────────────────────────────────────────

labor_df = spark.read.synapsesql("lh_retaildata.retail.vw_labor_coverage") \
    .filter(f"str_id = '{STORE_ID}'") \
    .toPandas()

# Type average headcount for benchmark
type_avg = spark.read.synapsesql("lh_retaildata.retail.vw_labor_coverage") \
    .filter(f"str_type = '{STORE_TYPE}'") \
    .groupBy("str_id") \
    .agg(F.countDistinct("emp_id").alias("headcount")) \
    .toPandas()
store_type_avg_headcount = round(type_avg["headcount"].mean(), 1) if not type_avg.empty else 0

# KPI 1: Headcount vs type average
headcount = labor_df["emp_id"].nunique() if not labor_df.empty else 0
is_understaffed = headcount < store_type_avg_headcount

# KPI 2: Avg orders managed per associate
avg_orders_per_assoc = round(labor_df["orders_managed"].mean(), 1) if not labor_df.empty else 0

# KPI 3: Single-associate shift count
if not labor_df.empty:
    labor_df["shift"] = pd.cut(
        pd.to_datetime(labor_df["shift_start"]).dt.hour,
        bins=[-1, 11, 16, 24], labels=["Morning", "Afternoon", "Evening"]
    )
    shift_counts = labor_df.groupby("shift")["emp_id"].nunique()
    single_associate_shifts = shift_counts[shift_counts == 1].index.tolist()
else:
    single_associate_shifts = []

labor_kpis = {"headcount": headcount, "store_type_avg": store_type_avg_headcount,
              "is_understaffed": is_understaffed, "avg_orders_per_assoc": avg_orders_per_assoc,
              "single_associate_shifts": single_associate_shifts}

print(f"Headcount: {headcount} (type avg: {store_type_avg_headcount}) {'⚠ UNDERSTAFFED' if is_understaffed else ''}")
print(f"Avg orders/associate: {avg_orders_per_assoc}")
if single_associate_shifts:
    print(f"⚠ Single-associate shifts: {', '.join(single_associate_shifts)}")

In [ ]:
# ── Inventory Metrics ──────────────────────────────────────────────────

store_inv = spark.read.synapsesql("lh_retaildata.retail.vw_store_inventory_health") \
    .filter(f"str_id = '{STORE_ID}'") \
    .toPandas()

# KPI 1: Out-of-stock count
oos_products = store_inv[store_inv["risk_flag"].isin(["OUT_OF_STOCK", "OUT_OF_STOCK_REORDER_PENDING"])]
oos_count = len(oos_products)

# KPI 2: Below safety stock count
below_safety_count = len(store_inv[store_inv["risk_flag"] == "BELOW_SAFETY_STOCK"])

# KPI 3: Shrinkage cost
shrinkage_cost = round((store_inv["shrinkage_qty"] * store_inv["unit_cost_amt"]).sum(), 2) if not store_inv.empty else 0

inv_kpis = {"total_products": len(store_inv), "oos_count": oos_count,
            "below_safety_stock": below_safety_count, "shrinkage_cost": shrinkage_cost}

print(f"Total products tracked: {len(store_inv)}")
print(f"Out of stock: {oos_count}")
print(f"Below safety stock: {below_safety_count}")
print(f"Shrinkage cost: ${shrinkage_cost:,.2f}")

In [ ]:
# ── Risk & Opportunity Detection ───────────────────────────────────────

risks = []
opportunities = []

# Sales risk: revenue decline
if rev_wow is not None and rev_wow <= -REVENUE_CRITICAL_DROP:
    risks.append({"severity": "CRITICAL", "domain": "Sales",
        "title": f"Revenue declined {abs(rev_wow)}% WoW",
        "detail": f"${revenue_prior:,.0f} → ${revenue:,.0f}",
        "dollar_impact": round(revenue_prior - revenue, 2),
        "reasoning": "Decline >15% signals significant disruption."})
elif rev_wow is not None and rev_wow <= -REVENUE_WARNING_DROP:
    risks.append({"severity": "WARNING", "domain": "Sales",
        "title": f"Revenue declined {abs(rev_wow)}% WoW",
        "detail": f"${revenue_prior:,.0f} → ${revenue:,.0f}",
        "dollar_impact": round(revenue_prior - revenue, 2),
        "reasoning": "Moderate decline warrants investigation."})

# Inventory risk: OOS
if oos_count > 0:
    risks.append({"severity": "WARNING", "domain": "Inventory",
        "title": f"{oos_count} products out of stock",
        "detail": "; ".join(oos_products.head(5)["prod_name"].tolist()),
        "dollar_impact": None,
        "reasoning": "OOS products directly reduce revenue and customer satisfaction."})

# Inventory risk: shrinkage
if shrinkage_cost > 0:
    risks.append({"severity": "MONITORING", "domain": "Inventory",
        "title": f"Shrinkage detected: ${shrinkage_cost:,.0f}",
        "detail": f"{len(store_inv[store_inv['risk_flag'] == 'SHRINKAGE_DETECTED'])} products affected",
        "dollar_impact": shrinkage_cost,
        "reasoning": "Shrinkage represents direct loss of inventory value."})

# Labor risk: understaffed
if is_understaffed:
    risks.append({"severity": "WARNING", "domain": "Labor",
        "title": f"Understaffed: {headcount} vs {store_type_avg_headcount} avg",
        "detail": f"{STORE_TYPE} stores average {store_type_avg_headcount} associates.",
        "dollar_impact": None,
        "reasoning": "Low staffing degrades service and shelf replenishment."})

# Opportunity: revenue gap vs peers
if revenue < peer_avg_revenue and peer_avg_revenue > 0:
    gap = round(peer_avg_revenue - revenue, 2)
    opportunities.append({"domain": "Sales",
        "title": f"Close ${gap:,.0f}/week gap vs peer average",
        "detail": f"Store: ${revenue:,.0f} vs peer avg: ${peer_avg_revenue:,.0f}",
        "estimated_dollar_value": gap,
        "reasoning": "Underperforming vs peers suggests execution or assortment opportunity."})

# Opportunity: add staff to high-revenue understaffed store
if is_understaffed and revenue > peer_avg_revenue:
    opportunities.append({"domain": "Labor",
        "title": f"Add {store_type_avg_headcount - headcount:.0f} associates",
        "detail": f"High revenue (${revenue:,.0f}) with below-avg staffing.",
        "estimated_dollar_value": None,
        "reasoning": "Protecting high revenue requires adequate staffing."})

risks.sort(key=lambda r: {"CRITICAL": 0, "WARNING": 1, "MONITORING": 2}.get(r["severity"], 3))

print(f"Risks: {len(risks)} | Opportunities: {len(opportunities)}")
for r in risks:
    icon = {"CRITICAL": "🔴", "WARNING": "🟡", "MONITORING": "🟢"}[r["severity"]]
    print(f"  {icon} {r['domain']}: {r['title']}")
for o in opportunities:
    print(f"  💡 {o['domain']}: {o['title']}")

In [ ]:
# ── LLM Narrative ──────────────────────────────────────────────────────

try:
    from openai import AzureOpenAI
    import os

    client = AzureOpenAI(
        azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT", "<your-endpoint>"),
        api_key=os.environ.get("AZURE_OPENAI_KEY", "<your-key>"),
        api_version="2024-12-01-preview",
    )

    payload = {
        "store": {"name": STORE_NAME, "id": STORE_ID, "type": STORE_TYPE, "region": REG_NAME},
        "period": f"{period_start} to {period_end}",
        "sales": sales_kpis, "labor": labor_kpis, "inventory": inv_kpis,
        "risks": risks, "opportunities": opportunities,
    }

    response = client.chat.completions.create(
        model=os.environ.get("AZURE_OPENAI_MODEL", "gpt-4o"),
        messages=[
            {"role": "system", "content": (
                "You are a retail analyst. Generate a concise store intelligence report with: "
                "1) Executive Summary (3 sentences), 2) Key Risks, 3) Opportunities, 4) Top 3 Recommended Actions. "
                "Use only the data provided. Be direct and specific."
            )},
            {"role": "user", "content": json.dumps(payload, indent=2, default=str)},
        ],
        temperature=0.3, max_tokens=1500,
    )
    llm_narrative = response.choices[0].message.content
    print("✓ LLM narrative generated.")

except Exception as e:
    print(f"⚠ LLM skipped: {e} — using rule-based summary.")
    parts = [f"## Executive Summary",
             f"{STORE_NAME} generated ${revenue:,.0f} ({'↑' if (rev_wow or 0) >= 0 else '↓'}{abs(rev_wow or 0)}% WoW). "
             f"Headcount: {headcount} ({'below' if is_understaffed else 'at/above'} avg). "
             f"{len(risks)} risks, {len(opportunities)} opportunities identified."]
    for r in risks:
        parts.append(f"\n**{r['severity']} — {r['domain']}**: {r['title']}\n{r['detail']}")
    for o in opportunities:
        parts.append(f"\n**Opportunity — {o['domain']}**: {o['title']}\n{o['detail']}")
    llm_narrative = "\n".join(parts)

In [ ]:
# ── Report Output ──────────────────────────────────────────────────────

from datetime import datetime as dt

report_html = f"""
<div style="font-family: Segoe UI, sans-serif; max-width:800px; margin:auto;">
<h1 style="border-bottom:3px solid #1E40AF; padding-bottom:8px;">Store Intelligence: {STORE_NAME}</h1>
<p style="color:#64748B;">{STORE_TYPE} | {REG_NAME} | {period_start} → {period_end}</p>

<table style="width:100%; border-collapse:collapse; margin:12px 0;">
<tr style="background:#1E40AF; color:white; font-weight:bold;">
  <td colspan="3" style="padding:6px;">Sales</td>
  <td colspan="3" style="padding:6px;">Labor</td>
  <td colspan="3" style="padding:6px;">Inventory</td>
</tr>
<tr style="font-size:15px;">
  <td style="padding:6px;">Rev: ${revenue:,.0f}</td>
  <td style="padding:6px; color:{'#22C55E' if (rev_wow or 0) >= 0 else '#EF4444'};">{rev_wow or 'N/A'}% WoW</td>
  <td style="padding:6px;">#{peer_rank}/{peer_count}</td>
  <td style="padding:6px; color:{'#EF4444' if is_understaffed else '#22C55E'};">{headcount}/{store_type_avg_headcount} staff</td>
  <td style="padding:6px;">{avg_orders_per_assoc} ord/assoc</td>
  <td style="padding:6px;">{len(single_associate_shifts)} thin shifts</td>
  <td style="padding:6px; color:{'#EF4444' if oos_count else '#22C55E'};">{oos_count} OOS</td>
  <td style="padding:6px;">{below_safety_count} low stock</td>
  <td style="padding:6px;">${shrinkage_cost:,.0f} shrink</td>
</tr>
</table>

<div style="line-height:1.6;">{llm_narrative}</div>
<p style="color:#94A3B8; font-size:11px;">Generated {dt.now().strftime('%Y-%m-%d %H:%M')}</p>
</div>
"""
displayHTML(report_html)

---
## Phase 7 — Persist Recommendations to Table

Writes all insights (risks, opportunities, correlations, executive summary) to `retail.recommendations` for the Recommendation Data Agent.

In [ ]:
# ── Persist Recommendations ────────────────────────────────────────────

from pyspark.sql.types import StructType, StructField, StringType, FloatType
from datetime import datetime as dt

report_date = today
generated_at = dt.now()

rows = []
for r in risks:
    rows.append({"report_date": str(report_date), "str_id": str(STORE_ID),
        "str_name": STORE_NAME, "str_type": STORE_TYPE, "reg_id": REG_ID, "reg_name": REG_NAME,
        "insight_type": "RISK", "severity": r["severity"], "domain": r["domain"],
        "title": r["title"], "detail": r["detail"], "reasoning": r["reasoning"],
        "dollar_impact": float(r["dollar_impact"]) if r.get("dollar_impact") else None,
        "generated_at": str(generated_at)})

for o in opportunities:
    rows.append({"report_date": str(report_date), "str_id": str(STORE_ID),
        "str_name": STORE_NAME, "str_type": STORE_TYPE, "reg_id": REG_ID, "reg_name": REG_NAME,
        "insight_type": "OPPORTUNITY", "severity": None, "domain": o["domain"],
        "title": o["title"], "detail": o["detail"], "reasoning": o["reasoning"],
        "dollar_impact": float(o["estimated_dollar_value"]) if o.get("estimated_dollar_value") else None,
        "generated_at": str(generated_at)})

rows.append({"report_date": str(report_date), "str_id": str(STORE_ID),
    "str_name": STORE_NAME, "str_type": STORE_TYPE, "reg_id": REG_ID, "reg_name": REG_NAME,
    "insight_type": "EXECUTIVE_SUMMARY", "severity": None, "domain": "All",
    "title": f"Daily Intelligence — {STORE_NAME}", "detail": llm_narrative,
    "reasoning": None, "dollar_impact": None, "generated_at": str(generated_at)})

kpi_json = json.dumps({"revenue": revenue, "revenue_wow": rev_wow, "orders": orders,
    "headcount": headcount, "peer_rank": peer_rank, "oos_count": oos_count,
    "below_safety_stock": below_safety_count, "shrinkage_cost": shrinkage_cost})
rows.append({"report_date": str(report_date), "str_id": str(STORE_ID),
    "str_name": STORE_NAME, "str_type": STORE_TYPE, "reg_id": REG_ID, "reg_name": REG_NAME,
    "insight_type": "KPI_SNAPSHOT", "severity": None, "domain": "All",
    "title": f"KPI Snapshot — {STORE_NAME} — {report_date}", "detail": kpi_json,
    "reasoning": None, "dollar_impact": None, "generated_at": str(generated_at)})

schema = StructType([
    StructField("report_date", StringType(), False), StructField("str_id", StringType(), True),
    StructField("str_name", StringType(), True), StructField("str_type", StringType(), True),
    StructField("reg_id", StringType(), True), StructField("reg_name", StringType(), True),
    StructField("insight_type", StringType(), False), StructField("severity", StringType(), True),
    StructField("domain", StringType(), True), StructField("title", StringType(), False),
    StructField("detail", StringType(), True), StructField("reasoning", StringType(), True),
    StructField("dollar_impact", FloatType(), True), StructField("generated_at", StringType(), False),
])

df_recs = spark.createDataFrame(rows, schema=schema)
spark.sql(f"DELETE FROM retail.recommendations WHERE str_id = '{STORE_ID}' AND report_date = '{report_date}'")
df_recs.write.mode("append").saveAsTable("retail.recommendations")

print(f"✓ Wrote {len(rows)} rows to retail.recommendations for {STORE_NAME} ({report_date})")